# Figure 5 — Real data and scale (S14 LARRY rows, S11, S20)

Sources: `benchmarks/synthetic_multimodal/s14/test.csv` (kind = larry; lineage Spearman,
five permutation nulls, held-out day-6 energy distances; both configurations),
`benchmarks/real_flow/{velocity_agreement,by_cell_type,operator_agreement}.csv` (S11),
`benchmarks/scaling/{scaling,agreement,larry_full_lambda1}.csv` (S20). LARRY barcodes are used for
scoring only.

In [1]:
from _style import *
import json as _json
from scipy.stats import spearmanr

prov = Provenance("Fig5", notebook="notebooks/fig5_real_scale.ipynb")
test = pd.read_csv(prov.use("benchmarks/synthetic_multimodal/s14/test.csv", "a", "S14"))
va = pd.read_csv(prov.use("benchmarks/real_flow/velocity_agreement.csv", "b", "S11"))
bct = pd.read_csv(prov.use("benchmarks/real_flow/by_cell_type.csv", "b", "S11"))
oa = pd.read_csv(prov.use("benchmarks/real_flow/operator_agreement.csv", "b", "S11"))
scal = pd.read_csv(prov.use("benchmarks/scaling/scaling.csv", "c", "S20"))
agr = pd.read_csv(prov.use("benchmarks/scaling/agreement.csv", "c", "S20"))
full = pd.read_csv(prov.use("benchmarks/scaling/larry_full_lambda1.csv", "d", "S20"))
V = prov.value
larry = test[test.kind == "larry"].set_index("configuration_id")

fig = plt.figure(figsize=(DOUBLE_COL, 5.9))
outer = fig.add_gridspec(2, 1, hspace=0.62, height_ratios=[1, 1])
g0 = outer[0].subgridspec(1, 5, width_ratios=[0.78, 0.78, 1.0, 0.5, 0.95], wspace=0.95)
g1 = outer[1].subgridspec(1, 4, width_ratios=[1, 1, 1, 1.05], wspace=0.72)

In [2]:
ax = fig.add_subplot(g0[0, 0])
for i, cid in enumerate(["reference", "selected"]):
    r = larry.loc[cid]
    nulls = _json.loads(r.lineage_nulls)
    ax.scatter(i + np.linspace(-0.12, 0.12, len(nulls)), nulls, color=C_BASE, s=8, zorder=2)
    ax.scatter([i], [r.lineage_spearman], color=C_REF if cid == "reference" else C_SELECTED, s=24, marker="D", zorder=3)
    V(f"a_{cid}_lineage_spearman", r.lineage_spearman); V(f"a_{cid}_null_max", max(nulls))
ax.set_xticks([0, 1]); ax.set_xticklabels(["refer-\nence", "selected"], fontsize=6); ax.set_xlim(-0.5, 1.5)
ax.set_ylabel("Spearman, flow-J λ₁ vs lineage\ndisplacement (◆) and 5 nulls (●)"); ax.set_ylim(0, 0.65)
ax.set_title("LARRY lineage\nagreement (S14)", fontsize=6.5)
panel_letter(ax, "a", x=-0.45)

ax = fig.add_subplot(g0[0, 1])
cats = [("heldout_flow_energy", "flow"), ("heldout_mean_displacement_energy", "mean\ndispl."), ("heldout_no_movement_energy", "no\nmove")]
w = 0.36
for i, cid in enumerate(["reference", "selected"]):
    r = larry.loc[cid]
    vals = [r[c] for c, _ in cats]
    cols = [C_REF if cid == "reference" else C_SELECTED, C_BASE, C_BASE]
    ax.bar(np.arange(3) + (i - 0.5) * w, vals, width=w * 0.92, color=cols,
           hatch=None if cid == "reference" else "////", edgecolor="white", lw=0)
    for c, _ in cats:
        V(f"a_{cid}_{c}", r[c])
ax.set_xticks(range(3)); ax.set_xticklabels([l.replace("\n", " ") for _, l in cats], rotation=40, ha="right", rotation_mode="anchor")
ax.set_ylabel("energy distance to held-out day 6")
ax.set_title("held-out day 6\n(hatched = selected)", fontsize=6.5)
prov.panel("a", "LARRY (S14 test): lineage Spearman vs five permutation nulls; held-out day-6 energy distance of flow vs two baselines, both configurations")

In [3]:
ax = fig.add_subplot(g0[0, 2])
pairs = [("larry", "ot", "lineage", "LARRY OT vs lineage", C_FLOW), ("scnt_hema", "labelling", "dynamo_gene", "scNT labelling vs dynamo", C_LABEL),
         ("scnt_hema", "labelling", "scvelo", "scNT labelling vs scVelo", C_SCVELO), ("scnt_hema", "scvelo", "dynamo_gene", "scNT scVelo vs dynamo", C_BASE)]
bins = np.linspace(-1, 1, 41)
for ds, a_, b_, lab, col in pairs:
    v = va[(va.dataset == ds) & (va.source_a == a_) & (va.source_b == b_)].velocity_cosine
    ax.hist(v, bins=bins, histtype="step", color=col, lw=1.0, density=True, label=f"{lab} (median {f2(v.median())})")
    V(f"b_velocity_cosine_median_{ds}_{a_}_{b_}", v.median()); V(f"b_velocity_n_{ds}_{a_}_{b_}", int(len(v)))
ax.set_xlabel("per-cell velocity cosine"); ax.set_ylabel("density")
ax.legend(fontsize=6, loc="upper right", bbox_to_anchor=(1.12, -0.25), ncol=2, columnspacing=1.0, handlelength=1.2)
ax.set_title("S11 velocity agreement", fontsize=6.5)
panel_letter(ax, "b", x=-0.3)

ax = fig.add_subplot(g0[0, 4])
L = bct[(bct.dataset == "larry") & (bct.operator == "measured")]
ot = L[L.source == "ot"].set_index("cell_type"); li = L[L.source == "lineage"].set_index("cell_type")
order = ot.sort_values("n_cells", ascending=False).index
yy = np.arange(len(order))
ax.scatter(ot.loc[order, "lambda1_median"], yy, color=C_FLOW, s=10, label="OT flow J", zorder=3)
ax.scatter(li.loc[order, "lambda1_median"], yy, color=C_LINEAGE, s=10, marker="s", label="lineage flow J", zorder=3)
ax.set_yticks(yy); ax.set_yticklabels([f"{c} ({ot.loc[c, 'n_cells']})" for c in order], fontsize=6)
ax.invert_yaxis(); ax.set_xlabel("median λ₁ (flow J)"); ax.axvline(0, color=C_BASE, lw=0.5)
ax.legend(fontsize=6, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, handletextpad=0.1, columnspacing=0.8)
ax.set_title("LARRY cell-type λ₁\n(S11; n = OT cells)", fontsize=6.2)
for src, t in [("ot", ot), ("lineage", li)]:
    V(f"b_{src}_undiff_gt_neu_and_mon", bool(t.loc["Undifferentiated", "lambda1_median"] > max(t.loc["Neutrophil", "lambda1_median"], t.loc["Monocyte", "lambda1_median"])))
    for c in ["Undifferentiated", "Neutrophil", "Monocyte"]:
        V(f"b_{src}_lambda1_{c}", t.loc[c, "lambda1_median"])
o = oa[(oa.dataset == "larry")]
V("b_ot_vs_lineage_lambda1_spearman", o[(o.comparison_kind == "independent_sources")].lambda1_spearman.iloc[0])
st = o[(o.left == "ot") & (o.right == "ot") & o.right_estimator.isin(["measured_s1", "measured_s2", "dynamo"])]
V("b_ot_seed_estimator_spearman_min", st.lambda1_spearman.min()); V("b_ot_seed_estimator_spearman_max", st.lambda1_spearman.max())
prov.panel("b", "S11: per-cell velocity cosine between sources; LARRY cell-type median lambda1 for OT vs lineage flow J")

In [4]:
sol_col = {"dense": C_BASE, "minibatch": C_FLOW, "lowrank_r50": OKABE["purple"], "lowrank_r200": OKABE["orange"]}
n_full = int(full.n_cells.sum())
V("c_n_cells_full", n_full)
def ncells(nd):
    return n_full if nd == "full" else 3 * int(nd)
scal["cells"] = scal.n_per_day.map(ncells)
ax_t = fig.add_subplot(g1[0, 0]); ax_m = fig.add_subplot(g1[0, 1])
for sol, col in sol_col.items():
    for stage, ls in [("ot", "-"), ("fit_and_jacobian", "--")]:
        h = scal[(scal.solver == sol) & (scal.stage == stage)].sort_values("cells")
        if len(h) == 0:
            continue
        ax_t.plot(h.cells, h.wall_seconds_external, color=col, ls=ls, marker="o", ms=2.5, lw=0.9)
        if stage == "ot":
            ax_m.plot(h.cells, h.max_rss_bytes / 1e9, color=col, marker="o", ms=2.5, lw=0.9, label=sol)
for ax in (ax_t, ax_m):
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("cells (3 days)")
ax_t.set_ylabel("wall time (s)"); ax_m.set_ylabel("peak RSS, OT stage (GB)")
ax_t.legend(handles=[plt.Line2D([], [], color="black", ls="-", label="OT"), plt.Line2D([], [], color="black", ls="--", label="drift fit + J")],
            fontsize=6, loc="upper left", handlelength=1.8)
ax_m.legend(fontsize=6, loc="upper left", handlelength=1.5)
ax_t.set_ylim(top=ax_t.get_ylim()[1] * 8); ax_m.set_ylim(top=ax_m.get_ylim()[1] * 8)
ax_t.set_title("S20 runtime", fontsize=6.5); ax_m.set_title("S20 memory", fontsize=6.5)
panel_letter(ax_t, "c", x=-0.4)
mf = scal[(scal.solver == "minibatch") & (scal.n_per_day == "full")].set_index("stage")
V("c_full_ot_seconds", mf.loc["ot", "wall_seconds_external"]); V("c_full_ot_rss_gb", mf.loc["ot", "max_rss_bytes"] / 1e9)
V("c_full_fit_seconds", mf.loc["fit_and_jacobian", "wall_seconds_external"]); V("c_full_fit_rss_gb", mf.loc["fit_and_jacobian", "max_rss_bytes"] / 1e9)
V("c_full_total_minutes", (mf.wall_seconds_external.sum()) / 60); V("c_full_peak_rss_gb", mf.max_rss_bytes.max() / 1e9)
V("c_dense_skipped_full", bool(((scal.solver == "dense") & (scal.n_per_day == "full")).sum() == 0))

ax = fig.add_subplot(g1[0, 2])
for sol in ["minibatch", "lowrank_r50", "lowrank_r200"]:
    h = agr[agr.solver == sol].sort_values("n_per_day")
    ax.plot(h.n_per_day * 3, h.median_velocity_cosine, color=sol_col[sol], marker="o", ms=3, lw=1, label=f"{sol} cosine")
    ax.plot(h.n_per_day * 3, h.flow_lambda1_spearman, color=sol_col[sol], marker="s", ms=3, lw=1, ls="--")
    for _, r in h.iterrows():
        V(f"c_agree_{sol}_{int(r.n_per_day)}_cosine", r.median_velocity_cosine)
        V(f"c_agree_{sol}_{int(r.n_per_day)}_lambda1", r.flow_lambda1_spearman)
ax.set_xscale("log"); ax.set_xlabel("cells (3 days)"); ax.set_ylabel("agreement with dense OT")
ax.set_xticks([6000, 12000, 24000]); ax.set_xticklabels(["6k", "12k", "24k"]); ax.minorticks_off()
ax.set_ylim(-0.15, 1.05); ax.axhline(0, color=C_BASE, lw=0.4)
ax.legend(handles=[plt.Line2D([], [], color=sol_col[s], lw=1.2, label=s) for s in ["minibatch", "lowrank_r50", "lowrank_r200"]]
          + [plt.Line2D([], [], color="black", marker="o", ms=3, label="cosine (median)"),
             plt.Line2D([], [], color="black", marker="s", ls="--", ms=3, label="λ₁ Spearman")], fontsize=6, loc="center left", handlelength=1.8)
ax.set_title("S20 agreement vs dense", fontsize=6.5)
ax.legend_.set_bbox_to_anchor((0.0, 0.45))
lr = agr[agr.solver.str.startswith("lowrank")]
V("c_lowrank_cosine_min", lr.median_velocity_cosine.min()); V("c_lowrank_cosine_max", lr.median_velocity_cosine.max())
V("c_lowrank_lambda1_absmax", lr.flow_lambda1_spearman.abs().max())
prov.panel("c", "S20: wall time and OT-stage peak memory vs cells (log-log, per solver); agreement with dense OT per solver and size")

In [5]:
ax = fig.add_subplot(g1[0, 3])
g = full.groupby("cell_type")
full_ct = g.apply(lambda x: np.average(x.lambda1_median, weights=x.n_cells), include_groups=False)
s11_ct = g.s11_lambda1_median.first()
rho = spearmanr(full_ct, s11_ct.loc[full_ct.index]).statistic
V("d_celltype_spearman_recomputed", rho); V("d_celltype_spearman_file", float(full.s11_cell_type_lambda1_spearman.iloc[0]))
V("d_n_cell_types", int(len(full_ct)))
ax.scatter(s11_ct.loc[full_ct.index], full_ct, color=C_FLOW, s=12, zorder=3)
lim = [0, max(full_ct.max(), s11_ct.max()) * 1.1]
ax.plot(lim, lim, color=C_BASE, lw=0.5, ls=":")
ax.set_xlim(-0.004, lim[1] * 1.02); ax.set_ylim(-0.004, lim[1] * 1.02)
repel_labels(ax, s11_ct.loc[full_ct.index].values, full_ct.values, list(full_ct.index), fontsize=6)
ax.set_xlabel("median λ₁, 4,000 cells/day (S11)"); ax.set_ylabel(f"λ₁, all {n_full:,} cells (S20 minibatch)")
ax.set_title(f"full LARRY vs subsample\n(Spearman {f2(rho)})", fontsize=6.2)
panel_letter(ax, "d", x=-0.4)
prov.panel("d", "Full-LARRY cell-type lambda1 (cell-weighted mean of per-day medians) vs S11 4,000/day cell-type median")
save(fig, "Fig5")

[save] main/Fig5: 169.1 x 133.4 mm, PNG 3995 x 3151 px @ 600 dpi


(PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig5.pdf'),
 PosixPath('/Users/terooatt/Documents/scJDO2-figures/manuscript/figures/main/Fig5.png'))

In [6]:
v = prov.values
prov.legend(f"""
**Figure 5 | Real data: lineage agreement, held-out prediction and scale (S14, S11, S20).** LARRY in vitro haematopoiesis
(GSE140802; days 2/4/6); dense analyses use a fixed sample of 4,000 cells per day (`default_rng(0)`); barcodes used for scoring only.
**a**, S14 held-out test (pre-registered; gate G3 applies to the selected configuration and failed). Left, Spearman between flow-J λ₁ and
lineage displacement (◆) vs five permutation nulls (●): reference {f2(v['a_reference_lineage_spearman'])} (null max {f2(v['a_reference_null_max'])}),
selected {f2(v['a_selected_lineage_spearman'])} (null max {f2(v['a_selected_null_max'])}). Right, energy distance between the day-6 distribution
predicted by integrating the fitted drift from day 4 and the held-out day 6: reference flow {f2(v['a_reference_heldout_flow_energy'])}, selected
flow {f2(v['a_selected_heldout_flow_energy'])}; mean-displacement baseline {f2(v['a_reference_heldout_mean_displacement_energy'])}, no-movement
baseline {f2(v['a_reference_heldout_no_movement_energy'])}.
**b**, S11 (pre-registered criteria; 6 of 14 met). Left, per-cell cosine between velocity sources: LARRY OT vs clonal displacement
(n = {v['b_velocity_n_larry_ot_lineage']} cells with a lineage velocity; median {f2(v['b_velocity_cosine_median_larry_ot_lineage'])}); scNT-seq haematopoiesis
labelling vs dynamo {f2(v['b_velocity_cosine_median_scnt_hema_labelling_dynamo_gene'])}, labelling vs scVelo {f2(v['b_velocity_cosine_median_scnt_hema_labelling_scvelo'])},
scVelo vs dynamo {f2(v['b_velocity_cosine_median_scnt_hema_scvelo_dynamo_gene'])} (n = {v['b_velocity_n_scnt_hema_labelling_scvelo']}). Right, LARRY cell-type
median λ₁ of the OT and lineage flow J (PCA-30); Undifferentiated > Neutrophil and Monocyte for both. Per-cell OT vs lineage λ₁ Spearman
{f2(v['b_ot_vs_lineage_lambda1_spearman'])} (null level); OT flow J across fit seeds 1–2 and dynamo SparseVFC {rng_str(v['b_ot_seed_estimator_spearman_min'], v['b_ot_seed_estimator_spearman_max'])}.
**c**, S20 (pre-registered gates G1, G2 passed): wall time (OT solid, drift fit + Jacobians dashed) and OT-stage peak memory vs total cells
(2,000–16,000 per day and all {v['c_n_cells_full']:,} cells) for dense, mini-batch (4,000-cell batches) and low-rank (moscot rank 50/200) OT;
dense OT on the full data was not run (cost matrix ≈ 21 GB). Full LARRY with mini-batch OT: OT {f2(v['c_full_ot_seconds'], 0)} s ({f2(v['c_full_ot_rss_gb'], 2)} GB),
drift fit + Jacobians {f2(v['c_full_fit_seconds'], 0)} s ({f2(v['c_full_fit_rss_gb'], 2)} GB). Right, agreement with dense OT: mini-batch median velocity cosine
{f2(v['c_agree_minibatch_8000_cosine'], 4)} and λ₁ Spearman {f2(v['c_agree_minibatch_8000_lambda1'], 4)} at 8,000 cells/day (the 4,000/day gate row is trivial: one batch = whole day);
low-rank cosine {rng_str(v['c_lowrank_cosine_min'], v['c_lowrank_cosine_max'])}, |λ₁ Spearman| ≤ {f2(v['c_lowrank_lambda1_absmax'])}.
**d**, Cell-type λ₁ on all cells (cell-count-weighted mean of per-day medians, {v['d_n_cell_types']} cell types) vs the 4,000-per-day S11 result: Spearman {f2(v['d_celltype_spearman_recomputed'])}.
""")
prov.write()